In [2]:
%run setup.py

OISCurve | currency=EUR | valuation_date=2026-03-24
NSSCurve | currency=EUR | valuation_date=2026-04-20


## 0 FX Rate Conventions

An FX rate is expressed as units of the **quote currency**
needed to buy one unit of the **base currency**:

$$\text{BASE/QUOTE} = \text{price of 1 BASE in QUOTE units}$$

**Market conventions:**

| Pair | Base | Quote | Convention |
|------|------|-------|------------|
| EUR/USD | EUR | USD | EUR always base vs USD |
| EUR/CHF | EUR | CHF | EUR base vs Swiss franc |
| GBP/USD | GBP | USD | GBP always base vs USD |
| USD/JPY | USD | JPY | USD base vs Asian currencies |
| USD/CHF | USD | CHF | USD base vs Swiss franc |
| USD/BRL | USD | BRL | USD base vs EM currencies |
| EUR/GBP | EUR | GBP | EUR base vs other European currencies |

EUR is the strongest base currency -- it is quoted as base against
almost all other currencies. USD is base against most Asian and
emerging market currencies but quote against EUR and GBP. CHF appears
on both sides depending on the pair -- EUR/CHF and USD/CHF both have
CHF as quote, reflecting the Swiss franc's role as a safe-haven
funding currency rather than a base.


For this notebook we price **EUR/USD** forwards, the most liquid
FX pair globally, with approximately USD 1.5 trillion traded daily.

# Asset Pricing: FX Forwards

An FX forward is an agreement to exchange two currencies at a
predetermined rate on a future date. No cash changes hands at
inception -- the exchange occurs at maturity at the agreed forward rate.

**Covered Interest Rate Parity (CIP):**

The forward rate is not a forecast -- it is an arbitrage condition.
If the forward rate deviated from CIP, a riskless profit would be
available by borrowing in one currency, converting spot, investing in
the other currency, and locking in the forward. In equilibrium:

$$F(0,T) = S_0 \cdot \frac{P_{EUR}(0,T)}{P_{USD}(0,T)}$$

Where:
- $F(0,T)$ -- forward EUR/USD rate for delivery at T
- $S_0$ -- spot EUR/USD rate today
- $P_{EUR}(0,T)$ -- EUR OIS discount factor to maturity T
- $P_{USD}(0,T)$ -- USD OIS (SOFR) discount factor to maturity T

In continuous compounding:

$$F(0,T) = S_0 \cdot e^{(r_{USD} - r_{EUR}) \cdot T}$$

The forward rate is higher than spot when USD rates exceed EUR rates
(USD at a discount, EUR at a premium) and lower when EUR rates exceed
USD rates.

**CIP deviation -- the basis:**

Post-2008 CIP does not hold exactly in practice. The deviation is
called the **cross-currency basis** -- the spread paid above or below
CIP to obtain funding in a foreign currency via the FX swap market.
A negative EUR/USD basis means it costs more to borrow USD via EUR
FX swaps than the CIP rate implies -- reflecting USD funding scarcity.

**Regulation context:**
- FRTB SA -- delta FX risk sensitivity per currency pair
- EMIR -- FX forwards with maturity > 3 days are OTC derivatives
  subject to reporting and margin requirements
- IFRS 9 -- FX forwards are Level 2 fair value instruments

## 1 FX and Interest Rate Parity -- Intuition

Consider a EUR-based investor with EUR 1M to invest for 1 year.
Two strategies must produce the same result in equilibrium:

**Strategy A -- invest in EUR:**
- Invest EUR 1M at the EUR risk-free rate $r_{EUR}$
- Receive EUR $1M \cdot e^{r_{EUR} \cdot T}$ at maturity

**Strategy B -- invest in USD:**
- Convert EUR 1M to USD at spot rate $S_0$: receive USD $1M \cdot S_0$
- Invest USD at the USD risk-free rate $r_{USD}$
- Receive USD $1M \cdot S_0 \cdot e^{r_{USD} \cdot T}$ at maturity
- Convert back to EUR at the forward rate $F(0,T)$
- Receive EUR $1M \cdot S_0 \cdot e^{r_{USD} \cdot T} / F(0,T)$

**No-arbitrage condition:**

Both strategies must produce the same EUR amount:

$$e^{r_{EUR} \cdot T} = S_0 \cdot e^{r_{USD} \cdot T} / F(0,T)$$

Solving for the forward rate:

$$F(0,T) = S_0 \cdot e^{(r_{USD} - r_{EUR}) \cdot T}$$

**What this means in practice:**

If USD rates are higher than EUR rates ($r_{USD} > r_{EUR}$), the
forward rate $F(0,T)$ is higher than the spot rate $S_0$ -- the USD
trades at a forward discount (you need more USD per EUR in the future).
If EUR rates are higher, the forward is below spot.

The forward rate is therefore not a market forecast of where EUR/USD
will be in one year -- it is a mathematical consequence of today's
interest rates. Any deviation would create a riskless arbitrage.

**Example with current rates:**

If EUR OIS 1Y = 2.00% and USD SOFR 1Y = 4.50% and spot = 1.0800:

$$F(0,1Y) = 1.0800 \cdot e^{(0.045 - 0.020) \cdot 1} = 1.0800 \cdot e^{0.025} = 1.1072$$

The EUR trades at a forward premium -- you get more USD per EUR in
one year because USD rates are higher and the market compensates EUR
holders for the lower EUR investment return.